# MolecularFunction → BiologicalProcess Relation Pipeline

Builds a unified, deduplicated edge table for the **MolecularFunction–BiologicalProcess** relation.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | kg_type | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- Monarch (MolecularActivity_BiologicalProcess)

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [22]:
import pandas as pd
import os

BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR     = BASE_DIR + 'processed_data/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/MOLECULARFUNCTION_BIOLOGICALPROCESS/ALL_MOLECULARFUNCTION_BIOLOGICALPROCESS.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]

## 1 · Mapping DBs

In [18]:
GO = pd.read_csv(MAPPING_DIR + 'MESH/MESH_GO_ID_NAME.csv')
GO_dict = dict(zip(GO['id'], GO['name']))
del GO

## 2 · Load Source Files

In [19]:
# 1. Monarch - MolecularActivity_BiologicalProcess (Native MF -> BP)
Monarch_MF_BP = pd.read_csv(PROC_DIR + 'Monarch/Human/MolecularActivity_BiologicalProcess_Monarch.csv')
Monarch_MF_BP.columns = Monarch_MF_BP.columns.str.lower()
if 'kgsource' in Monarch_MF_BP.columns:
    Monarch_MF_BP = Monarch_MF_BP.rename(columns={'kgsource': 'kg_source'})
if 'head_name' in Monarch_MF_BP.columns:
    Monarch_MF_BP = Monarch_MF_BP.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in Monarch_MF_BP.columns:
    Monarch_MF_BP = Monarch_MF_BP.rename(columns={'tail_name': 'tail_detail_name'})
required_columns = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]
Monarch_MF_BP = Monarch_MF_BP[required_columns]
Monarch_MF_BP['head_id_is'] = 'GO'
Monarch_MF_BP['tail_id_is'] = 'GO'
Monarch_MF_BP['relation'] = 'MolecularFunction_BiologicalProcess'
Monarch_MF_BP['head_type'] = 'MolecularFunction'
Monarch_MF_BP['tail_type'] = 'BiologicalProcess'
Monarch_MF_BP['kg_type'] = 'Generalised'
print(f"Monarch Native MF->BP: {len(Monarch_MF_BP):,} rows")
Monarch_MF_BP

Monarch Native MF->BP: 848 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type
0,GO:0061459,MolecularFunction_BiologicalProcess,GO:1903826,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,L-arginine transmembrane transporter activity,L-arginine transmembrane transport,Generalised
1,GO:0061542,MolecularFunction_BiologicalProcess,GO:0006744,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,3-demethylubiquinol-n 3-O-methyltransferase ac...,ubiquinone biosynthetic process,Generalised
2,GO:0061607,MolecularFunction_BiologicalProcess,GO:0061606,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,peptide alpha-N-propionyltransferase activity,N-terminal protein amino acid propionylation,Generalised
3,GO:0061608,MolecularFunction_BiologicalProcess,GO:0051170,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,nuclear import signal receptor activity,import into nucleus,Generalised
4,GO:0061686,MolecularFunction_BiologicalProcess,GO:0140479,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,hercynylcysteine sulfoxide synthase activity,ergothioneine biosynthesis from histidine via ...,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...
843,GO:0052650,MolecularFunction_BiologicalProcess,GO:0042572,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,all-trans-retinol dehydrogenase (NADP+) activity,retinol metabolic process,Generalised
844,GO:0052807,MolecularFunction_BiologicalProcess,GO:0046223,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,aflatoxin reductase (coenzyme F420) activity,aflatoxin catabolic process,Generalised
845,GO:0052885,MolecularFunction_BiologicalProcess,GO:0042572,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,"all-trans-retinyl-ester hydrolase, 11-cis reti...",retinol metabolic process,Generalised
846,GO:0060175,MolecularFunction_BiologicalProcess,GO:0031547,MolecularFunction,related_to,BiologicalProcess,MonarchKG,GO,GO,brain-derived neurotrophic factor receptor act...,brain-derived neurotrophic factor receptor sig...,Generalised


## 3 · GO Term Normalisation & NA Audit Before Deduplication

In [20]:
# Fill detail names using MESH GO dictionary
final_df = Monarch_MF_BP
head_detail = final_df['head'].map(GO_dict)
final_df['head_detail_name'] = final_df['head_detail_name'].fillna(head_detail)

tail_detail = final_df['tail'].map(GO_dict)
final_df['tail_detail_name'] = final_df['tail_detail_name'].fillna(tail_detail)

final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)

print(f"Consolidated rows (before deduplication): {len(final_df):,}")
print("\nNaN audit before deduplication:")
print(final_df.isna().sum())

Consolidated rows (before deduplication): 848

NaN audit before deduplication:
head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
kg_type             0
dtype: int64


## 6 · Save Output

In [23]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 848 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MOLECULARFUNCTION_BIOLOGICALPROCESS/ALL_MOLECULARFUNCTION_BIOLOGICALPROCESS.csv
